In [1]:
import zipfile
import os
import rasterio
import numpy as np
import pandas as pd

# 1. Extract archive (2).zip
with zipfile.ZipFile('archive (2).zip', 'r') as zip_ref:
    zip_ref.extractall('.')

print("Extraction complete!")

# 2. Parse extracted .tif files into a clean columnar DataFrame
records = []

for filename in os.listdir('.'):
    if filename.endswith('.tif'):
        parts = filename.replace('.tif', '').split('_')
        if len(parts) >= 2:
            index_type = parts[0]
            date_str = parts[1].split('(')[0]

            with rasterio.open(filename) as src:
                arr = src.read(1, masked=True)
                mean_val = float(np.nanmean(arr)) if arr.size > 0 else np.nan

            records.append({
                'date': pd.to_datetime(date_str, errors='coerce'),
                'index_type': index_type,
                'mean_value': mean_val
            })

# 3. Pivot into columnar format (Index Types as Columns)
df_columns = pd.DataFrame(records).dropna(subset=['date']).pivot_table(
    index='date',
    columns='index_type',
    values='mean_value',
    aggfunc='mean'
).reset_index().sort_values('date')

df_columns.head(15)

Extraction complete!


KeyError: ['date']

In [2]:
import zipfile
import os
import rasterio
import numpy as np
import pandas as pd

# 1. Extract archive (2).zip
with zipfile.ZipFile('archive (2).zip', 'r') as zip_ref:
    zip_ref.extractall('.')

print("Extraction complete!")

# 2. Walk through all directories to find .tif files
records = []

for root, dirs, files in os.walk('.'):
    for filename in files:
        if filename.endswith('.tif'):
            filepath = os.path.join(root, filename)
            parts = filename.replace('.tif', '').split('_')

            if len(parts) >= 2:
                index_type = parts[0]
                date_str = parts[1].split('(')[0]

                with rasterio.open(filepath) as src:
                    arr = src.read(1, masked=True)
                    mean_val = float(np.nanmean(arr)) if arr.size > 0 else np.nan

                records.append({
                    'date': pd.to_datetime(date_str, errors='coerce'),
                    'index_type': index_type,
                    'mean_value': mean_val
                })

# 3. Create DataFrame and pivot into columnar format
df = pd.DataFrame(records)

if df.empty:
    print("No .tif files were found. Please check your folder structure.")
else:
    df_columns = df.dropna(subset=['date']).pivot_table(
        index='date',
        columns='index_type',
        values='mean_value',
        aggfunc='mean'
    ).reset_index().sort_values('date')

    display(df_columns.head(15))

Extraction complete!
No .tif files were found. Please check your folder structure.


In [3]:
import os

# Check all files inside the Dataset folder
dataset_files = []
for root, dirs, files in os.walk('Dataset'):
    for file in files:
        dataset_files.append(os.path.join(root, file))

print(f"Total files found in Dataset folder: {len(dataset_files)}")
print("First 10 files:")
for f in dataset_files[:10]:
    print(f)

Total files found in Dataset folder: 30
First 10 files:
Dataset/Traininig/drone-samples/0.jpg
Dataset/Traininig/drone-samples/1.jpg
Dataset/Traininig/glass bottle-samples/0.jpg
Dataset/Traininig/papercoke cup-samples/0.jpg
Dataset/Traininig/papercoke cup-samples/1.jpg
Dataset/Traininig/plastic water bottle-samples/0.jpg
Dataset/Traininig/plastic water bottle-samples/1.jpg
Dataset/Traininig/Fanta can/0.jpg
Dataset/Traininig/diustbin-samples/0.jpg
Dataset/Traininig/Iron Hammer-samples/0.jpg


In [4]:
import os
import rasterio
import numpy as np
import pandas as pd

records = []
valid_extensions = ('.tif', '.TIF', '.tiff', '.TIFF')

for root, dirs, files in os.walk('.'):
    for filename in files:
        if filename.endswith(valid_extensions):
            filepath = os.path.join(root, filename)

            # Extract filename without extension
            base_name = os.path.splitext(filename)[0]
            parts = base_name.split('_')

            if len(parts) >= 2:
                index_type = parts[0]
                date_str = parts[1].split('(')[0]

                try:
                    with rasterio.open(filepath) as src:
                        arr = src.read(1, masked=True)
                        mean_val = float(np.nanmean(arr)) if arr.size > 0 else np.nan

                    records.append({
                        'filepath': filepath,
                        'date': pd.to_datetime(date_str, errors='coerce'),
                        'index_type': index_type,
                        'mean_value': mean_val
                    })
                except Exception as e:
                    print(f"Error reading {filename}: {e}")

df = pd.DataFrame(records)

if df.empty:
    print("No matching raster files found.")
else:
    # Pivot into columnar format
    df_columns = df.dropna(subset=['date']).pivot_table(
        index='date',
        columns='index_type',
        values='mean_value',
        aggfunc='mean'
    ).reset_index().sort_values('date')

    display(df_columns.head(15))

No matching raster files found.


In [5]:
# Print column summary matrix
summary_stats = df_columns.describe().T[['count', 'mean', 'std', 'min', '50%', 'max']]
summary_stats.columns = ['Count', 'Mean', 'Std Dev', 'Min', 'Median', 'Max']
display(summary_stats.round(4))

NameError: name 'df_columns' is not defined

In [6]:
# Compute time interval between observations
df_gaps = df_columns[['date']].copy()
df_gaps['days_since_last_pass'] = df_gaps['date'].diff().dt.days

print(f"Average days between passes: {df_gaps['days_since_last_pass'].mean():.1f} days")
print(f"Maximum gap between passes: {df_gaps['days_since_last_pass'].max():.0f} days")

display(df_gaps.sort_values('days_since_last_pass', ascending=False).head(10))

NameError: name 'df_columns' is not defined

In [7]:
import os
import pandas as pd

dataset_records = []

# Walk through the Dataset folder to gather JPG image details
for root, dirs, files in os.walk('Dataset'):
    for file in files:
        if file.lower().endswith(('.jpg', '.jpeg', '.png')):
            full_path = os.path.join(root, file)

            # Extract category name from parent folder
            folder_name = os.path.basename(root)

            # Get file size in KB
            file_size_kb = round(os.path.getsize(full_path) / 1024, 2)

            dataset_records.append({
                'filename': file,
                'category': folder_name,
                'filepath': full_path,
                'size_kb': file_size_kb
            })

# Create DataFrame
df_images = pd.DataFrame(dataset_records)

# 1. Display Class Counts Breakdown (Columns format)
print("--- Category Breakdown ---")
category_summary = df_images['category'].value_counts().reset_index()
category_summary.columns = ['Category Name', 'Image Count']
display(category_summary)

# 2. Display First 15 Image Records
print("\n--- First 15 Records ---")
display(df_images[['category', 'filename', 'size_kb', 'filepath']].head(15))

--- Category Breakdown ---


,Category Name,Image Count
0,drone-samples,4
1,papercoke cup-samples,4
2,plastic water bottle-samples,4
3,tin can-samples,4
4,plastic bottle-samples,4
5,glass bottle-samples,2
6,diustbin-samples,2
7,Class 8-samples,2
8,Iron Hammer-samples,2
9,Fanta can,1



--- First 15 Records ---


,category,filename,size_kb,filepath
0,drone-samples,0.jpg,10.46,Dataset/Traininig/drone-samples/0.jpg
1,drone-samples,1.jpg,14.23,Dataset/Traininig/drone-samples/1.jpg
2,glass bottle-samples,0.jpg,12.14,Dataset/Traininig/glass bottle-samples/0.jpg
3,papercoke cup-samples,0.jpg,16.35,Dataset/Traininig/papercoke cup-samples/0.jpg
4,papercoke cup-samples,1.jpg,16.35,Dataset/Traininig/papercoke cup-samples/1.jpg
5,plastic water bottle-samples,0.jpg,8.38,Dataset/Traininig/plastic water bottle-samples...
6,plastic water bottle-samples,1.jpg,9.93,Dataset/Traininig/plastic water bottle-samples...
7,Fanta can,0.jpg,4.47,Dataset/Traininig/Fanta can/0.jpg
8,diustbin-samples,0.jpg,11.11,Dataset/Traininig/diustbin-samples/0.jpg
9,Iron Hammer-samples,0.jpg,11.75,Dataset/Traininig/Iron Hammer-samples/0.jpg


In [8]:
from PIL import Image

image_dims = []

for idx, row in df_images.iterrows():
    try:
        with Image.open(row['filepath']) as img:
            width, height = img.size
            mode = img.mode  # RGB, L (grayscale), etc.

        image_dims.append({
            'filename': row['filename'],
            'category': row['category'],
            'width': width,
            'height': height,
            'aspect_ratio': round(width / height, 2),
            'mode': mode
        })
    except Exception as e:
        print(f"Error opening {row['filepath']}: {e}")

df_dims = pd.DataFrame(image_dims)

print("--- Image Dimensions Overview ---")
display(df_dims.head(15))

--- Image Dimensions Overview ---


,filename,category,width,height,aspect_ratio,mode
0,0.jpg,drone-samples,224,224,1.0,RGB
1,1.jpg,drone-samples,224,224,1.0,RGB
2,0.jpg,glass bottle-samples,224,224,1.0,RGB
3,0.jpg,papercoke cup-samples,224,224,1.0,RGB
4,1.jpg,papercoke cup-samples,224,224,1.0,RGB
5,0.jpg,plastic water bottle-samples,224,224,1.0,RGB
6,1.jpg,plastic water bottle-samples,224,224,1.0,RGB
7,0.jpg,Fanta can,224,224,1.0,RGB
8,0.jpg,diustbin-samples,224,224,1.0,RGB
9,0.jpg,Iron Hammer-samples,224,224,1.0,RGB


In [9]:
import numpy as np

color_features = []

for idx, row in df_images.iterrows():
    try:
        with Image.open(row['filepath']) as img:
            img_rgb = img.convert('RGB')
            arr = np.array(img_rgb)

            # Mean intensity per color channel
            r_mean = round(float(np.mean(arr[:, :, 0])), 2)
            g_mean = round(float(np.mean(arr[:, :, 1])), 2)
            b_mean = round(float(np.mean(arr[:, :, 2])), 2)

        color_features.append({
            'category': row['category'],
            'filename': row['filename'],
            'red_mean': r_mean,
            'green_mean': g_mean,
            'blue_mean': b_mean
        })
    except Exception as e:
        print(f"Error processing {row['filepath']}: {e}")

df_colors = pd.DataFrame(color_features)

print("--- Tabular Channel Intensity Means ---")
display(df_colors.head(15))

--- Tabular Channel Intensity Means ---


,category,filename,red_mean,green_mean,blue_mean
0,drone-samples,0.jpg,139.72,149.39,56.52
1,drone-samples,1.jpg,146.90,158.40,64.62
2,glass bottle-samples,0.jpg,130.68,163.98,176.53
3,papercoke cup-samples,0.jpg,184.84,110.38,101.12
4,papercoke cup-samples,1.jpg,184.84,110.38,101.12
5,plastic water bottle-samples,0.jpg,192.66,182.88,101.47
6,plastic water bottle-samples,1.jpg,172.82,166.42,103.58
7,Fanta can,0.jpg,217.52,159.26,90.03
8,diustbin-samples,0.jpg,109.19,130.40,139.13
9,Iron Hammer-samples,0.jpg,102.59,166.40,170.58


In [10]:
category_colors = df_colors.groupby('category')[['red_mean', 'green_mean', 'blue_mean']].mean().round(2)
category_colors.columns = ['Avg Red', 'Avg Green', 'Avg Blue']

print("--- Average RGB Intensities Per Class ---")
display(category_colors)

--- Average RGB Intensities Per Class ---


,Avg Red,Avg Green,Avg Blue
category,,,
Class 8-samples,217.52,159.26,90.03
Fanta can,217.52,159.26,90.03
Iron Hammer-samples,102.59,166.40,170.58
diustbin-samples,109.19,130.40,139.13
drone-samples,143.31,153.89,60.57
fanta can,217.52,159.26,90.03
glass bottle-samples,130.68,163.98,176.53
papercoke cup-samples,184.84,110.38,101.12
plastic bottle-samples,148.04,123.36,84.84


In [11]:
# Create a cleaned category column
df_images['clean_category'] = (
    df_images['category']
    .str.lower()
    .str.replace('-samples', '', regex=False)
    .str.strip()
)

print("--- Cleaned Category Breakdown ---")
cleaned_summary = df_images['clean_category'].value_counts().reset_index()
cleaned_summary.columns = ['Cleaned Category', 'Total Count']
display(cleaned_summary)

--- Cleaned Category Breakdown ---


,Cleaned Category,Total Count
0,drone,4
1,papercoke cup,4
2,tin can,4
3,plastic water bottle,4
4,plastic bottle,4
5,glass bottle,2
6,diustbin,2
7,fanta can,2
8,iron hammer,2
9,class 8,2


In [12]:
from sklearn.model_selection import train_test_split

# Split dataset records 80/20 grouped by class
train_df, val_df = train_test_split(
    df_images,
    test_size=0.2,
    stratify=df_images['clean_category'],
    random_state=42
)

# Display tabular split summary
split_summary = pd.DataFrame({
    'Train Count': train_df['clean_category'].value_counts(),
    'Val Count': val_df['clean_category'].value_counts()
}).fillna(0).astype(int)

print("--- Dataset Train/Validation Split Counts ---")
display(split_summary)

ValueError: The test_size = 6 should be greater or equal to the number of classes = 10